# Warehouse March Export
>This notebook was creared to export, aggregate and clean warehouse data from the hugginface releasefor March 2026.

In [1]:
%pip -q install duckdb huggingface_hub

import os, duckdb
from dotenv import load_dotenv
from pathlib import Path


#Resolve the repository root directory by looking for the data/raw/content_refresh_anonymized.csv file
repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    data_file = candidate / 'data' / 'raw' / 'content_refresh_anonymized.csv'
    if data_file.exists():
        repo_root = candidate
        break

output_path = repo_root / 'work' / 'outputs' / 'features.parquet'
output_path.parent.mkdir(parents=True, exist_ok=True)

load_dotenv()
HF_TOKEN = os.environ.get('HF_TOKEN')

Note: you may need to restart the kernel to use updated packages.


In [2]:
con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"

feature_aggregation_query = f"""
WITH feature_aggregation AS (
    SELECT 
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions,
        SUM(gsc_clicks) AS total_clicks,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS avg_position,
        COUNT(DISTINCT report_date) AS active_days,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) AS historical_ctr,
        SUM(ga4_pageviews) AS total_pageviews,
        SUM(ga4_sessions) AS total_sessions,
        SUM(ga4_engaged_sessions) / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,
        SUM(ga4_total_engagement_sec) / NULLIF(SUM(ga4_users), 0) AS avg_engagement_time_per_user,
        SUM(sessions_organic) / NULLIF(SUM(ga4_sessions), 0) AS organic_session_ratio,
        SUM(ai_chatgpt + ai_perplexity + ai_gemini + ai_copilot + ai_claude + ai_meta + ai_other) AS ai_referral_sessions
    FROM {REL}
    WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
)
SELECT * FROM feature_aggregation

"""


#Because a python string literal cannot contain single quotes, we need to escape them in the output path for the SQL query
output_sql_path = str(output_path).replace("'", "''")
con.execute(f"COPY ({feature_aggregation_query}) TO '{output_sql_path}' (FORMAT PARQUET, COMPRESSION ZSTD)")

row_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_sql_path}')").fetchone()[0]
print(f"Successfully exported {row_count:,} March 2026 rows to {output_path}")
print(con.execute(feature_aggregation_query).df())

Successfully exported 331,437 March 2026 rows to C:\Users\DELL\Documents\flyrank-ml-internship-starter\work\outputs\features.parquet
                 client_hash_id           content_hash_id  total_impressions  \
0       client_625b6439094e23e4  content_8c121255eb885630                0.0   
1       client_625b6439094e23e4  content_77c52144b33b0dba                0.0   
2       client_625b6439094e23e4  content_39fa7500959586bb                0.0   
3       client_625b6439094e23e4  content_2ec2ece77555de8e                0.0   
4       client_625b6439094e23e4  content_017d01d662487698                0.0   
...                         ...                       ...                ...   
331432  client_fef1a8f436438636  content_2e785a6773e4f652                0.0   
331433  client_fef1a8f436438636  content_a94094011d5e242e                0.0   
331434  client_fef1a8f436438636  content_67b2f0de2b5425ab                0.0   
331435  client_fef1a8f436438636  content_584b12f73046103a          